In [ ]:
# ============================================================
# 2. Environment and configuration
# ============================================================
from pathlib import Path
import json, re, warnings, math
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

sns.set_theme(context="notebook", style="whitegrid", font_scale=1.0)

# Point this at the matrix output produced by run_matrix().
# Change only this path when moving the notebook to another machine.
PROJECT_ROOT = Path(".")   # e.g. Path("/Volumes/Amirali/hidden_states/experiments/...")
RESULT_ROOT = Path("/Volumes/Amirali/hidden_states")
# Common filenames produced by the probing pipeline.
CSV_GLOBS = [
    "**/*layer_probe_results.csv",
    "**/probe_matrix_checkpoint/per_entry_results/*.csv",
]

# Optional: a directory containing individual run manifests / metrics.json files.
MANIFEST_GLOB = "**/complete_run_metadata.json"
METRICS_GLOB = "**/metrics.json"

print("RESULT_ROOT =", RESULT_ROOT.resolve())

In [ ]:
# ============================================================
# 3. Load every available probe-result table
# ============================================================
def find_result_csvs(root: Path):
    paths = []
    for pat in CSV_GLOBS:
        paths.extend(root.glob(pat))
    return sorted(set(p for p in paths if p.is_file()))

csv_paths = find_result_csvs(RESULT_ROOT)

if csv_paths:
    print(f"Found {len(csv_paths)} result CSV(s).")
    for p in csv_paths[:20]:
        print(" •", p)
else:
    print("No result CSVs found yet.")
    print("Set RESULT_ROOT to the matrix/experiment directory, then rerun this cell.")

frames = []
for p in csv_paths:
    try:
        df = pd.read_csv(p)
        df["_source_csv"] = str(p)
        frames.append(df)
    except Exception as exc:
        warnings.warn(f"Could not read {p}: {exc}")

results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print("Combined shape:", results.shape)
display(results.head(10) if not results.empty else pd.DataFrame())

In [ ]:
# ============================================================
# 5. Schema audit
# ============================================================
if results.empty:
    print("No data to inspect.")
else:
    cols = list(results.columns)
    print(f"{len(cols)} columns")
    display(pd.DataFrame({
        "column": cols,
        "dtype": [str(results[c].dtype) for c in cols],
        "non_null": [int(results[c].notna().sum()) for c in cols],
        "unique": [int(results[c].nunique(dropna=True)) for c in cols],
    }).sort_values(["column"]).reset_index(drop=True))

In [ ]:
# ============================================================
# 6. Run-level audit helpers
# ============================================================
def classify_status(row):
    # Friendly heuristic from available columns.
    if "status" in row.index and isinstance(row["status"], str):
        return row["status"]
    return "completed"

if not results.empty:
    # Identify likely model/dataset/layer columns.
    model_col = next((c for c in ["model", "model_name"] if c in results.columns), None)
    data_col = next((c for c in ["dataset", "dataset_name"] if c in results.columns), None)
    layer_col = next((c for c in ["layer", "layer_idx", "layer_index", "layer_name"] if c in results.columns), None)

    print("Detected:")
    print(" model column :", model_col)
    print(" dataset col  :", data_col)
    print(" layer column :", layer_col)

    if model_col and data_col:
        run_table = (
            results.groupby([model_col, data_col], dropna=False)
            .agg(rows=("__dummy__" if "__dummy__" in results else results.columns[0], "size"))
            .reset_index()
        )
        display(run_table)

In [ ]:
# ============================================================
# 7. Choose a primary metric without changing the experiment
# ============================================================
METRIC_PRIORITY = [
    "test_macro_f1",
    "test_balanced_accuracy",
    "test_micro_f1",
    "test_weighted_f1",
    "test_accuracy",
]

present_metrics = [m for m in METRIC_PRIORITY if m in results.columns]
print("Available headline metrics:", present_metrics)

PRIMARY_METRIC = present_metrics[0] if present_metrics else None
if PRIMARY_METRIC:
    print("Primary plotting metric:", PRIMARY_METRIC)
else:
    print("No standard headline metric found; inspect the schema above.")

In [ ]:
# ============================================================
# 8. Layer-wise trajectory plot
# ============================================================
if not results.empty and PRIMARY_METRIC:
    plot_df = results.copy()

    layer_candidates = ["layer", "layer_idx", "layer_index"]
    layer_col = next((c for c in layer_candidates if c in plot_df.columns), None)
    if layer_col is None:
        raise ValueError("Could not find a layer index column.")

    model_col = next((c for c in ["model", "model_name"] if c in plot_df.columns), None)
    data_col = next((c for c in ["dataset", "dataset_name"] if c in plot_df.columns), None)

    group_cols = [c for c in [model_col, data_col] if c]
    if group_cols:
        plot_df[PRIMARY_METRIC] = pd.to_numeric(plot_df[PRIMARY_METRIC], errors="coerce")
        summary = (
            plot_df.dropna(subset=[PRIMARY_METRIC, layer_col])
            .groupby(group_cols + [layer_col], as_index=False)[PRIMARY_METRIC]
            .mean()
        )

        plt.figure(figsize=(13, 7))
        sns.lineplot(
            data=summary,
            x=layer_col,
            y=PRIMARY_METRIC,
            hue=model_col,
            style=data_col if data_col else None,
            markers=True,
            linewidth=2.2,
        )
        plt.title(f"Layer-wise target recoverability — {PRIMARY_METRIC}")
        plt.xlabel("Hidden-state layer")
        plt.ylabel(PRIMARY_METRIC)
        plt.tight_layout()
        plt.show()

In [ ]:
# ============================================================
# 9. Detect shuffled-control columns and compute an advantage
# ============================================================
control_patterns = [
    "shuffle", "shuffled", "random", "control",
]

control_cols = [
    c for c in results.columns
    if any(p in c.lower() for p in control_patterns)
    and pd.api.types.is_numeric_dtype(results[c])
]

print("Potential shuffled/control metrics:")
for c in control_cols:
    print(" •", c)

# Optional manual override if your exact schema uses a known field.
SHUFFLE_METRIC = next((c for c in control_cols if "macro_f1" in c.lower()), None)
print("Chosen shuffle metric:", SHUFFLE_METRIC)

In [ ]:
# ============================================================
# 10. Real-vs-shuffle advantage
# ============================================================
if not results.empty and PRIMARY_METRIC and SHUFFLE_METRIC:
    layer_col = next((c for c in ["layer", "layer_idx", "layer_index"] if c in results.columns), None)
    model_col = next((c for c in ["model", "model_name"] if c in results.columns), None)
    data_col = next((c for c in ["dataset", "dataset_name"] if c in results.columns), None)

    tmp = results[[c for c in [layer_col, model_col, data_col, PRIMARY_METRIC, SHUFFLE_METRIC] if c]].copy()
    tmp["advantage"] = (
        pd.to_numeric(tmp[PRIMARY_METRIC], errors="coerce")
        - pd.to_numeric(tmp[SHUFFLE_METRIC], errors="coerce")
    )

    group_cols = [c for c in [model_col, data_col, layer_col] if c]
    tmp = tmp.groupby(group_cols, as_index=False)["advantage"].mean()

    plt.figure(figsize=(13, 7))
    sns.lineplot(
        data=tmp, x=layer_col, y="advantage",
        hue=model_col if model_col else None,
        style=data_col if data_col else None,
        markers=True, linewidth=2.2
    )
    plt.axhline(0, linewidth=1)
    plt.title("Recoverability advantage over the shuffled-label control")
    plt.xlabel("Hidden-state layer")
    plt.ylabel("Observed − shuffled")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 11. Compare probe families
# ============================================================
probe_tokens = ["logistic", "mlp", "linear"]
probe_columns = {}

for token in probe_tokens:
    probe_columns[token] = [
        c for c in results.columns
        if token in c.lower() and c.lower().endswith(("macro_f1", "micro_f1", "accuracy"))
    ]

print("Detected probe-related columns:")
for k, v in probe_columns.items():
    print(k, "→", v[:10])

In [ ]:
# ============================================================
# 12. Flexible-probe gain, when schema exposes it
# ============================================================
# If your CSV stores probe name in a column, this block becomes the preferred route.
probe_col = next((c for c in ["probe", "probe_name", "probe_type"] if c in results.columns), None)
layer_col = next((c for c in ["layer", "layer_idx", "layer_index"] if c in results.columns), None)

if probe_col and layer_col and PRIMARY_METRIC:
    tmp = results.copy()
    tmp[PRIMARY_METRIC] = pd.to_numeric(tmp[PRIMARY_METRIC], errors="coerce")

    pivot = tmp.pivot_table(
        index=layer_col, columns=probe_col, values=PRIMARY_METRIC, aggfunc="mean"
    )

    display(pivot)

    # Plot all available probe types
    plt.figure(figsize=(12, 7))
    for probe_name in pivot.columns:
        plt.plot(pivot.index, pivot[probe_name], marker="o", linewidth=2, label=str(probe_name))
    plt.title(f"Probe-family comparison — {PRIMARY_METRIC}")
    plt.xlabel("Hidden-state layer")
    plt.ylabel(PRIMARY_METRIC)
    plt.legend(title="Probe")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 13. Automatically discover geometry metrics
# ============================================================
geometry_cols = [
    c for c in results.columns
    if any(t in c.lower() for t in ["silhouette", "pca", "explained_variance"])
]

print("Geometry-related columns found:", geometry_cols)

if geometry_cols:
    display(results[geometry_cols].head())

In [ ]:
# ============================================================
# 14. Geometry heatmap
# ============================================================
if not results.empty:
    layer_col = next((c for c in ["layer", "layer_idx", "layer_index"] if c in results.columns), None)
    if layer_col and PRIMARY_METRIC:
        candidates = [c for c in [PRIMARY_METRIC] + geometry_cols if c in results.columns]
        if len(candidates) >= 2:
            heat = results.groupby(layer_col)[candidates].mean(numeric_only=True)

            plt.figure(figsize=(11, max(4, 0.55 * len(candidates))))
            sns.heatmap(heat.T, annot=True, fmt=".3f", cmap="vlag", center=0)
            plt.title("Layer × diagnostic metric map")
            plt.xlabel("Layer")
            plt.ylabel("Metric")
            plt.tight_layout()
            plt.show()

In [ ]:
# ============================================================
# 15. Find per-class metric files
# ============================================================
# The probing code writes metrics.json under:
# models/<probe>/<layer>/<repeat>/metrics.json
metric_paths = sorted(RESULT_ROOT.glob(METRICS_GLOB))
print(f"Found {len(metric_paths)} metrics.json files.")

# Read a small sample to inspect the schema.
samples = []
for p in metric_paths[:10]:
    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
        samples.append((str(p), obj))
    except Exception:
        pass

for path, obj in samples[:3]:
    print("\n", path)
    print("top-level keys:", list(obj)[:20])

In [ ]:
# ============================================================
# 16. Extract class-level F1 where available
# ============================================================
def find_class_metrics(obj):
    candidates = []
    if isinstance(obj, dict):
        for key, val in obj.items():
            if isinstance(val, dict):
                # Typical nested format: {"class_name": {"f1": ...}}
                for cls, metrics in val.items():
                    if isinstance(metrics, dict):
                        for mname in ["f1", "f1_score", "precision", "recall"]:
                            if mname in metrics and isinstance(metrics[mname], (int, float)):
                                candidates.append((str(cls), mname, float(metrics[mname])))
    return candidates

rows = []
for p in metric_paths:
    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
        found = find_class_metrics(obj)
        for cls, metric, value in found:
            rows.append({"path": str(p), "class": cls, "metric": metric, "value": value})
    except Exception:
        continue

class_metrics = pd.DataFrame(rows)
print("Class-level records:", class_metrics.shape)
display(class_metrics.head(20))

In [ ]:
# ============================================================
# 17. Model × layer heatmap
# ============================================================
if not results.empty and PRIMARY_METRIC:
    model_col = next((c for c in ["model", "model_name"] if c in results.columns), None)
    layer_col = next((c for c in ["layer", "layer_idx", "layer_index"] if c in results.columns), None)

    if model_col and layer_col:
        tmp = results[[model_col, layer_col, PRIMARY_METRIC]].copy()
        tmp[PRIMARY_METRIC] = pd.to_numeric(tmp[PRIMARY_METRIC], errors="coerce")
        mat = tmp.pivot_table(index=model_col, columns=layer_col, values=PRIMARY_METRIC, aggfunc="mean")

        plt.figure(figsize=(14, max(5, 0.55 * len(mat))))
        sns.heatmap(mat, annot=True, fmt=".3f", cmap="mako", cbar_kws={"label": PRIMARY_METRIC})
        plt.title(f"Model × layer recoverability — {PRIMARY_METRIC}")
        plt.xlabel("Layer")
        plt.ylabel("Model")
        plt.tight_layout()
        plt.show()

In [ ]:
# ============================================================
# 18. Best-layer summary table
# ============================================================
if not results.empty and PRIMARY_METRIC:
    model_col = next((c for c in ["model", "model_name"] if c in results.columns), None)
    data_col = next((c for c in ["dataset", "dataset_name"] if c in results.columns), None)
    layer_col = next((c for c in ["layer", "layer_idx", "layer_index"] if c in results.columns), None)

    if model_col and layer_col:
        tmp = results.copy()
        tmp[PRIMARY_METRIC] = pd.to_numeric(tmp[PRIMARY_METRIC], errors="coerce")
        keys = [c for c in [model_col, data_col] if c]

        summary = []
        for key, g in tmp.dropna(subset=[PRIMARY_METRIC]).groupby(keys):
            idx = g[PRIMARY_METRIC].idxmax()
            r = g.loc[idx]
            summary.append({
                **({keys[0]: key[0]} if len(keys) > 1 else {keys[0]: key}),
                **({keys[1]: key[1]} if len(keys) > 1 else {}),
                "best_layer": r[layer_col],
                "best_score": r[PRIMARY_METRIC],
            })

        best_layer_table = pd.DataFrame(summary).sort_values("best_score", ascending=False)
        display(best_layer_table)

In [ ]:
# ============================================================
# 19. Repeated-run uncertainty, when repeats exist
# ============================================================
repeat_col = next((c for c in ["repeat", "repeat_id", "seed"] if c in results.columns), None)
layer_col = next((c for c in ["layer", "layer_idx", "layer_index"] if c in results.columns), None)

if repeat_col and layer_col and PRIMARY_METRIC:
    tmp = results.copy()
    tmp[PRIMARY_METRIC] = pd.to_numeric(tmp[PRIMARY_METRIC], errors="coerce")

    grouped = (
        tmp.groupby([layer_col], as_index=False)[PRIMARY_METRIC]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    grouped["sem"] = grouped["std"] / np.sqrt(grouped["count"].clip(lower=1))

    plt.figure(figsize=(13, 7))
    plt.plot(grouped[layer_col], grouped["mean"], marker="o", linewidth=2.2)
    plt.fill_between(
        grouped[layer_col],
        grouped["mean"] - grouped["sem"],
        grouped["mean"] + grouped["sem"],
        alpha=0.18,
    )
    plt.title(f"Mean ± SEM across available repeats — {PRIMARY_METRIC}")
    plt.xlabel("Layer")
    plt.ylabel(PRIMARY_METRIC)
    plt.tight_layout()
    plt.show()
else:
    print("No repeat identifier detected; do not infer a variance estimate from this run.")

In [ ]:
# ============================================================
# 20. Diagnostic dashboard data
# ============================================================
# Build a compact layer-level diagnostic frame from whatever is present.
if not results.empty:
    layer_col = next((c for c in ["layer", "layer_idx", "layer_index"] if c in results.columns), None)
    diag_cols = [c for c in [
        PRIMARY_METRIC,
        "test_balanced_accuracy",
        "test_micro_f1",
        "test_macro_f1",
        "test_mcc",
        "test_hamming_score",
        "test_macro_jaccard",
    ] if c in results.columns]

    if layer_col and diag_cols:
        dashboard = results.groupby(layer_col)[diag_cols].mean(numeric_only=True)
        display(dashboard.round(4))

In [ ]:
# ============================================================
# 25. FINAL EXECUTIVE SUMMARY TABLE
# ============================================================
if not results.empty and PRIMARY_METRIC:
    model_col = next((c for c in ["model", "model_name"] if c in results.columns), None)
    data_col = next((c for c in ["dataset", "dataset_name"] if c in results.columns), None)
    layer_col = next((c for c in ["layer", "layer_idx", "layer_index"] if c in results.columns), None)

    if model_col and layer_col:
        tmp = results.copy()
        tmp[PRIMARY_METRIC] = pd.to_numeric(tmp[PRIMARY_METRIC], errors="coerce")

        keys = [model_col] + ([data_col] if data_col else [])
        rows = []

        for key, g in tmp.dropna(subset=[PRIMARY_METRIC]).groupby(keys):
            if not isinstance(key, tuple):
                key = (key,)
            best_idx = g[PRIMARY_METRIC].idxmax()
            best = g.loc[best_idx]

            row = {
                model_col: key[0],
                "best_layer": best[layer_col],
                "best_score": float(best[PRIMARY_METRIC]),
                "n_layers": int(g[layer_col].nunique()),
            }
            if data_col:
                row[data_col] = key[1]
            rows.append(row)

        final_summary = pd.DataFrame(rows).sort_values("best_score", ascending=False)
        display(final_summary.style.format({"best_score": "{:.4f}"}))
    else:
        print("Insufficient columns for a model-level summary.")